In [1]:
import torch
import numpy as np
from sklearn.datasets import load_diabetes, load_iris, load_wine, load_digits
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, CrossEntropyLoss, Dropout
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

In [2]:
class EarlyStopping:
    def __init__(self, patience=20, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = model.state_dict()
            self.counter = 0

In [3]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [4]:
def train_model_class(features, targets, skf, batch, epochs, criterion):
    accuracy_scores = []
    f1_scores = []
    conf_matrix = []

    for i, (train_index, test_index) in enumerate(skf.split(features, targets)):
        print(f"Treinando modelo ----- SKFold {i + 1}")

        # 1. Separar os dados brutos do fold atual
        X_train_raw = features[train_index]
        X_test_raw = features[test_index]
        y_train = targets[train_index]
        y_test = targets[test_index]

        # 2. Normalizar estritamente com base nos dados de treino deste fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_raw)
        X_test_scaled = scaler.transform(X_test_raw)  # Apenas transform no teste!

        train_data = MyDataset(X_train_scaled, y_train)
        train_loader = DataLoader(train_data, batch_size=batch, shuffle=True)

        x_test_tensor = Tensor(X_test_scaled).float()
        y_test_tensor = Tensor(y_test).long()
        y_true = y_test

        model= build_model() 
        optmizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

        early_stopping = EarlyStopping(patience=30, min_delta=0.0001)

        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                y = y.long()
                optmizer.zero_grad()
                output = model(x)
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()


        model.eval()
        with torch.no_grad():
            val_outputs = model(x_test_tensor)
            val_loss = criterion(val_outputs, y_test_tensor).item()

        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping ativado na época {i_epochs + 1}!")
            break

        model.load_state_dict(early_stopping.best_model_state)

        model.eval()
        with torch.no_grad():
            outputs = model(x_test_tensor)
            result = torch.argmax(outputs, dim=1).cpu().numpy()
         
            accuracy = accuracy_score(y_true, result)
            f1 = f1_score(y_true, result, average="macro")
            confusion = confusion_matrix(y_true, result)

            accuracy_scores.append(accuracy)
            conf_matrix.append(confusion)
            f1_scores.append(f1)
            
            print(f"Accuracy: {accuracy}")
            print(f"f1: {f1}")
            print(f"confusion: \n{confusion}")

    print(f"\nResultado modelo ----- Accuracy: {np.mean(accuracy_scores)}, f1: {np.mean(f1_scores)}")

In [5]:
dataset = load_digits()
features = dataset.data
targets = dataset.target

In [6]:
def build_model():
    return Sequential(
        Linear(64, 128),
        ReLU(),
        torch.nn.Dropout(0.2),
        Linear(128,64),
        ReLU(),
        Linear(64, 10)
)

In [7]:
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 16
skf = StratifiedKFold(n_splits=5, shuffle=True)

In [8]:
train_model_class(features, targets, skf, batch, max_epochs, criterion)

Treinando modelo ----- SKFold 1
Accuracy: 0.9833333333333333
f1: 0.9832073353362182
confusion: 
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 36  0  0  0  0  0  0  0  0]
 [ 0  0 36  0  0  0  0  0  0  0]
 [ 0  0  0 36  0  0  0  0  0  0]
 [ 0  0  0  0 37  0  0  0  0  0]
 [ 0  0  0  0  0 36  0  0  0  0]
 [ 0  0  0  0  1  0 35  0  0  0]
 [ 0  0  0  0  0  0  0 35  0  1]
 [ 0  0  0  0  0  0  0  1 34  0]
 [ 0  0  0  0  0  1  0  1  1 33]]
Treinando modelo ----- SKFold 2
Accuracy: 0.9833333333333333
f1: 0.9834612604210491
confusion: 
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 36  0  0  0  0  0  0  0  0]
 [ 0  1 35  0  0  0  0  0  0  0]
 [ 0  0  0 36  0  0  0  0  0  0]
 [ 0  0  0  0 36  0  0  0  0  0]
 [ 0  0  0  0  0 36  0  0  0  1]
 [ 0  1  0  0  0  0 35  0  0  0]
 [ 0  0  0  0  0  0  0 36  0  0]
 [ 0  2  0  0  0  0  0  0 33  0]
 [ 0  0  0  0  0  0  0  1  0 35]]
Treinando modelo ----- SKFold 3
Accuracy: 0.9693593314763231
f1: 0.9687172545584662
confusion: 
[[36  0  0  0  0  0  0  0  0  0]
 [ 0 35  0  0  0 